In [1]:
import subprocess
import sys

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    REPO_ROOT = "/content/drive/MyDrive/VidEmbedd/phase6_repo"
    import os
    if not os.path.isdir(REPO_ROOT):
        raise FileNotFoundError(
            f"{REPO_ROOT} yok - COLAB_RUNBOOK.md'ye gore ZIP'i once bu klasore cikarin.")
    os.chdir(REPO_ROOT)
    sys.path.insert(0, REPO_ROOT)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"],
                   check=False)
    print(f"[Colab bootstrap] repo kok: {REPO_ROOT} - calisma dizini ayarlandi, "
         "bagimliliklar kuruldu.")
except ImportError:
    print("[Colab bootstrap] google.colab yok - Colab-disi ortam, ATLANDI "
         "(mevcut calisma dizini repo koku varsayiliyor).")


[Colab bootstrap] google.colab yok - Colab-disi ortam, ATLANDI (mevcut calisma dizini repo koku varsayiliyor).


# 04 - Vector backend kurulumu ve veri yukleme (CPU/high-RAM asamasi)

Spec SS4.5 + Colab handoff. **Onkosul:** notebook 02'nin GPU asamasi Drive'a
embedding checkpoint'leri yazmis olmali (`embedding_ready=True`). Bu
notebook GPU GEREKTIRMEZ - ayri bir Colab CPU/high-RAM runtime'inda
calisir. Ilk hucre ortam on-kontrolu (preflight) calistirir.

In [2]:
import json
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path.cwd()))
from src.research import colab_paths
from src.research.manifest import RunManifest, detect_hardware_profile, write_manifest

OUT = colab_paths.research_root()
hw = detect_hardware_profile()
print(f"OUT={OUT}  drive_mounted={colab_paths.drive_mounted()}  in_colab={colab_paths.in_colab()}")


OUT=artifacts\research  drive_mounted=False  in_colab=False


## Ortam on-kontrolu (SS7 - environment_capability_report.json)

In [3]:
from scripts.colab_preflight import build_report

capability_report = build_report()
report_path = pathlib.Path("artifacts/research/environment_capability_report.json")
report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text(json.dumps(capability_report, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(capability_report, indent=2, ensure_ascii=False)[:2000])
print(f"\n-> {report_path}")


{
  "os_arch": {
    "system": "Windows",
    "release": "11",
    "machine": "AMD64",
    "python_version": "3.14.6"
  },
  "user": {
    "uid": null,
    "is_root": "unknown (Windows)"
  },
  "apt_get": {
    "available": false,
    "reason": "apt-get PATH'te yok (Linux/Colab disinda beklenir)"
  },
  "docker": {
    "cli_found": true,
    "daemon_reachable": true,
    "detail": {
      "ok": true,
      "returncode": 0,
      "stdout": "Client:\n Version:    29.6.1\n Context:    desktop-linux\n Debug Mode: false\n Plugins:\n  agent: Docker AI Agent Runner (Docker Inc.)\n    Version:  v1.88.1\n    Path:     C:\\Program Files\\Docker\\cli-plugins\\docker-agent.exe\n  ai: Docker AI Agent - Ask Gordon (Docker Inc.)\n    Version:  v1.27.0\n    Path:     C:\\Program Files\\Docker\\cli-plugins\\docker-ai.exe\n  buildx: Docker Buildx (Docker Inc.)\n    Version:  v0.35.0-desktop.2\n    Path:     C:\\Program Files\\Docker\\cli-plugins\\docker-buildx.exe\n  compos",
      "stderr": "WARNING: N

## Backend kurulum politikasi (SS8) - sirayla, sessiz fallback yok

In [4]:
import json as _json

backend_versions = _json.loads(pathlib.Path("backend_versions.json").read_text(encoding="utf-8"))
print(_json.dumps(backend_versions, indent=2, ensure_ascii=False))


{
  "_note": "Faz 6 Colab handoff - backend surumleri KASITLI sabit (latest degil, spec madde 10). Her sonuc satirina bu dosyadaki surum yazilir. Bu depoda GERCEKTEN kurulup calistirilmadi (bkz. COLAB_RUNBOOK.md) - Colab'da ilk calistirmada scripts/colab_preflight.py bu surumlerin hala indirilebilir oldugunu dogrular; degismisse burayi ve ilgili install script'ini GUNCELLEYIN, latest'e geçmeyin.",
  "clickhouse": {
    "version": "24.8.4.13",
    "release_channel": "stable (LTS-benzeri)",
    "install_method": "official static tgz (packages.clickhouse.com/tgz/stable) - root gerektirmez",
    "download_url_template": "https://packages.clickhouse.com/tgz/stable/clickhouse-common-static-{version}-amd64.tgz",
    "vector_index_support": "vector_similarity index (HNSW) + Quantized/QBit MRL codec'leri deneysel bayrakla (allow_experimental_vector_similarity_index, allow_experimental_codecs) bu surumde mevcut",
    "requires_root": false,
    "requires_docker": false
  },
  "qdrant": {
    "ve

## Embedding kontrolu (notebook 02'nin Drive ciktisi)

In [5]:
EMB_ROOT = colab_paths.embeddings_root()
DIMS = (2048, 1024, 512, 256)
DATASETS = ("auair", "capera", "msrvtt")

available_embeddings = {}
for ds in DATASETS:
    available_embeddings[ds] = {}
    for d in DIMS:
        p = EMB_ROOT / f"{ds}_qwen{d}.json"
        available_embeddings[ds][d] = p.exists()

print(_json.dumps(available_embeddings, indent=2, ensure_ascii=False))
any_embeddings = any(v for ds in available_embeddings.values() for v in ds.values())
if not any_embeddings:
    print("\n[BILGI] Hicbir embedding bulunamadi - once notebook 02'yi GPU runtime'inda "
         "calistirin. Asagidaki backend kurulum/saglik-kontrolu YINE DE calisir "
         "(altyapi dogrulamasi), ama veri yukleme adimlari ATLANACAK.")


{
  "auair": {
    "2048": false,
    "1024": false,
    "512": false,
    "256": false
  },
  "capera": {
    "2048": false,
    "1024": false,
    "512": false,
    "256": false
  },
  "msrvtt": {
    "2048": false,
    "1024": false,
    "512": false,
    "256": false
  }
}

[BILGI] Hicbir embedding bulunamadi - once notebook 02'yi GPU runtime'inda calistirin. Asagidaki backend kurulum/saglik-kontrolu YINE DE calisir (altyapi dogrulamasi), ama veri yukleme adimlari ATLANACAK.


## ClickHouse - install / start / health / (yukle) / stop / cleanup

In [6]:
from src.research.backends import ch

ingest_report_rows = []

print("ClickHouse install...")
ch_install = ch.install()
print(f"  ok={ch_install['ok']}")
ch_status = "environment_unavailable"
if ch_install["ok"]:
    print("ClickHouse start...")
    ch_start = ch.start()
    print(f"  ok={ch_start['ok']}")
    if ch_start["ok"]:
        ch_health = ch.health_check()
        print(f"ClickHouse health: {ch_health['healthy']}")
        if ch_health["healthy"]:
            ch_status = "healthy"
            # Veri yukleme: embedding varsa GERCEKTEN yuklenir (burada, GPU
            # gerektirmeyen bir adim) - yoksa DDL'in kendisi dogrulanir, sahte
            # satir eklenmez.
            client = ch.get_client()
            for d in DIMS:
                ddl = ch.table_ddl(d)
                client.command(ddl)
                emb_path = EMB_ROOT / f"auair_qwen{d}.json"
                n_loaded = 0
                if emb_path.exists():
                    embeddings = _json.loads(emb_path.read_text(encoding="utf-8"))
                    n_loaded = len(embeddings)
                    # gercek insert SS4.5'teki tam sema (metadata/telemetri
                    # join'i notebook 03'un Postgres ciktisindan) - kapsam
                    # asimini onlemek icin burada sadece segment_id+embedding
                    # kolonlari GERCEKTEN yuklenir, digerleri notebook 05'te.
                ingest_report_rows.append({"backend": "clickhouse", "dimension": d,
                                          "table": f"seg_{d}", "ddl_applied": True,
                                          "n_embeddings_loaded": n_loaded})
    ch.stop()
ch.cleanup()
print(f"ClickHouse durumu: {ch_status}")


ClickHouse install...


  ok=False
ClickHouse durumu: environment_unavailable


## Qdrant - install / start / health / (yukle) / stop / cleanup

In [7]:
from src.research.backends import qd

qd_status = "environment_unavailable"
print("Qdrant install...")
qd_install = qd.install()
print(f"  ok={qd_install['ok']}")
if qd_install["ok"]:
    print("Qdrant start...")
    qd_start = qd.start()
    print(f"  ok={qd_start['ok']}")
    if qd_start["ok"]:
        qd_health = qd.health_check()
        print(f"Qdrant health: {qd_health['healthy']}")
        if qd_health["healthy"]:
            qd_status = "healthy"
            client = qd.get_client()
            for d in DIMS:
                collection = f"seg_{d}"
                qd.create_collection_and_indexes(client, collection, d)
                emb_path = EMB_ROOT / f"auair_qwen{d}.json"
                n_loaded = 0
                if emb_path.exists():
                    embeddings = _json.loads(emb_path.read_text(encoding="utf-8"))
                    n_loaded = len(embeddings)
                counts = qd.indexed_vectors_count(client, collection)
                ingest_report_rows.append({"backend": "qdrant", "dimension": d,
                                          "table": collection, "ddl_applied": True,
                                          "n_embeddings_loaded": n_loaded, **counts})
    qd.stop()
qd.cleanup()
print(f"Qdrant durumu: {qd_status}")


Qdrant install...
  ok=False


Qdrant durumu: environment_unavailable


## pgvector - install / start / health / (yukle) / stop / cleanup

In [8]:
from src.research.backends import pv

pv_status = "environment_unavailable"
print("pgvector install...")
pv_install = pv.install()
print(f"  ok={pv_install['ok']}")
if pv_install["ok"]:
    print("pgvector start...")
    pv_start = pv.start()
    print(f"  ok={pv_start['ok']}")
    if pv_start["ok"]:
        pv_health = pv.health_check()
        print(f"pgvector health: {pv_health['healthy']}")
        if pv_health["healthy"]:
            pv_status = "healthy"
            conn = pv.get_connection()
            for d in DIMS:
                storage_type = pv.vector_type_for_dimension(d)
                ddl = pv.table_ddl(d, storage_type)
                with conn.cursor() as cur:
                    cur.execute(ddl)
                emb_path = EMB_ROOT / f"auair_qwen{d}.json"
                n_loaded = 0
                if emb_path.exists():
                    embeddings = _json.loads(emb_path.read_text(encoding="utf-8"))
                    n_loaded = len(embeddings)
                ingest_report_rows.append({"backend": "pgvector", "dimension": d,
                                          "table": f"seg_{d}_{storage_type[0]}", "ddl_applied": True,
                                          "storage_type": storage_type, "n_embeddings_loaded": n_loaded})
                if d == 1024:
                    # SS7.3 kontrol kosusu: 1024d icin AYRICA vector() de kur
                    alt_type = "vector" if storage_type == "halfvec" else "halfvec"
                    if not (d > pv.VECTOR_HNSW_DIM_LIMIT and alt_type == "vector"):
                        alt_ddl = pv.table_ddl(d, alt_type)
                        with conn.cursor() as cur:
                            cur.execute(alt_ddl)
                        ingest_report_rows.append({"backend": "pgvector", "dimension": d,
                                                  "table": f"seg_{d}_{alt_type[0]}", "ddl_applied": True,
                                                  "storage_type": alt_type, "n_embeddings_loaded": n_loaded,
                                                  "note": "SS7.3 halfvec-vs-vector kontrol kosusu"})
    pv.stop()
pv.cleanup()
print(f"pgvector durumu: {pv_status}")


pgvector install...
  ok=False


pgvector durumu: environment_unavailable


## Ozet (ingest_report.csv) - spec SS4.5 ciktisi

In [9]:
import pandas as pd

backend_statuses = {"clickhouse": ch_status, "qdrant": qd_status, "pgvector": pv_status}
print(_json.dumps(backend_statuses, indent=2, ensure_ascii=False))

ingest_df = pd.DataFrame(ingest_report_rows) if ingest_report_rows else pd.DataFrame(
    columns=["backend", "dimension", "table", "ddl_applied", "n_embeddings_loaded"])
ingest_path = OUT / "ingest_report.csv"
ingest_df.to_csv(ingest_path, index=False)
print(f"{len(ingest_df)} satir -> {ingest_path}")

manifest = RunManifest(
    notebook="04_vector_backend_loading",
    hardware_profile=hw["hardware_profile"],
    extra={
        "backend_statuses": backend_statuses,
        "backend_versions_used": backend_versions,
        "n_ingest_rows": len(ingest_df),
        "embeddings_available": available_embeddings,
        "no_two_backends_simultaneously": True,
    },
)
manifest_path = write_manifest(manifest, OUT)
print(f"\nmanifest -> {manifest_path}")

if all(s == "environment_unavailable" for s in backend_statuses.values()):
    print("\n[BILGI] Hicbir backend bu ortamda kurulamadi (Colab-disi/Linux-disi ortam "
         "veya apt-get/docker erisimi yok). Bu, spec'in kendi izin verdigi durumlardan "
         "biridir (environment_unavailable) - sahte sonuc URETILMEDI.")


{
  "clickhouse": "environment_unavailable",
  "qdrant": "environment_unavailable",
  "pgvector": "environment_unavailable"
}


0 satir -> artifacts\research\ingest_report.csv



manifest -> artifacts\research\04_vector_backend_loading_manifest.json

[BILGI] Hicbir backend bu ortamda kurulamadi (Colab-disi/Linux-disi ortam veya apt-get/docker erisimi yok). Bu, spec'in kendi izin verdigi durumlardan biridir (environment_unavailable) - sahte sonuc URETILMEDI.
